In [ ]:
%pip install -q "psycopg[binary]>=3.2,<4"

In [ ]:
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

import pandas as pd
import psycopg
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, PermissionDenied
from psycopg import sql

w = WorkspaceClient()
defaults = {
    "project_id": "medium-cdc-202611",
    "source_schema": "medium_cdc_20261123",
    "cdf_catalog": "main",
    "cdf_schema": "medium_cdc_20261123",
    "warehouse_id": "",
    "reader_client_id": "",
    "reader_secret_scope": "",
    "reader_secret_key": "",
}
for name, default in defaults.items():
    if name not in dbutils.widgets.getAll():
        dbutils.widgets.text(name, default)
p = dbutils.widgets.getAll()
for name in ("source_schema", "cdf_catalog", "cdf_schema"):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", p[name]):
        raise ValueError(f"{name} must be a simple SQL identifier")
warehouse = p["warehouse_id"]
if not warehouse:
    candidates = [s.id for s in w.warehouses.list() if s.id and s.state.value in {"RUNNING", "STARTING", "STOPPED"}]
    if len(candidates) != 1:
        raise ValueError("Select the warehouse_id for the CDC history query")
    warehouse = candidates[0]
pg_api = "/api/2.0/postgres/"
branch = f"projects/{p['project_id']}/branches/production"
db_resource = branch + "/databases/databricks-postgres"
endpoint = branch + "/endpoints/primary"
host = w.api_client.do("GET", pg_api + endpoint)["status"]["hosts"]["host"]
feeds = w.api_client.do("GET", pg_api + db_resource + "/cdf-configs").get("cdf_configs", [])
if not any((f.get("postgres_schema"), f.get("catalog"), f.get("schema")) ==
           (p["source_schema"], p["cdf_catalog"], p["cdf_schema"]) for f in feeds):
    raise ValueError("Enable Lakebase CDF for the selected source schema and Delta destination first")
history = f"`{p['cdf_catalog']}`.`{p['cdf_schema']}`.`lb_orders_history`"
try:
    api_info = w.api_client.do("GET", pg_api + db_resource + "/data-api")
except NotFound as error:
    raise ValueError("Enable the Lakebase Data API for this database and expose public, then rerun.") from error
api_url = (api_info.get("status", {}).get("url") or "").rstrip("/")
if not api_url:
    raise ValueError("Enable the Lakebase Data API for this database and wait for its URL.")
if urlsplit(api_url).scheme != "https" or urlsplit(api_url).hostname != host:
    raise ValueError("Unexpected Lakebase Data API host")
reader = w
if p["reader_client_id"]:
    reader = WorkspaceClient(host=w.config.host, client_id=p["reader_client_id"],
                             client_secret=dbutils.secrets.get(p["reader_secret_scope"], p["reader_secret_key"]),
                             auth_type="oauth-m2m")
http = requests.Session()
try:
    token = reader.postgres.generate_database_credential(endpoint=endpoint).token
except PermissionDenied as error:
    raise ValueError("Grant the Data API reader CAN USE on the Lakebase project, then rerun.") from error
http.headers["Authorization"] = "Bearer " + token
source_table = sql.Identifier(p["source_schema"], "orders")
def connect():
    return psycopg.connect(host=host, dbname="databricks_postgres", user=w.current_user.me().user_name,
                           password=w.postgres.generate_database_credential(endpoint=endpoint).token,
                           sslmode="require")

In [ ]:
with connect() as db:
    source_name = source_table.as_string(db)
    identity = db.execute("SELECT relreplident FROM pg_class WHERE oid=to_regclass(%s)", (source_name,)).fetchone()
    if identity != ("f",):
        raise ValueError(f"Enable REPLICA IDENTITY FULL on {source_name} for native Lakebase CDF")
    trigger = db.execute("""SELECT 1 FROM pg_trigger
        WHERE tgrelid=to_regclass(%s) AND tgname='orders_capture_change' AND tgenabled IN ('O','A')
        AND tgfoid=to_regprocedure('public.capture_order_change()')""", (source_name,)).fetchone()
    if trigger is None:
        db.execute(sql.SQL("LOCK TABLE {} IN SHARE ROW EXCLUSIVE MODE").format(source_table))
        other = db.execute("""SELECT tgrelid::regclass::text FROM pg_trigger
            WHERE tgfoid=to_regprocedure('public.capture_order_change()')
            AND tgrelid <> to_regclass(%s) AND NOT tgisinternal LIMIT 1""", (source_name,)).fetchone()
        if other:
            raise ValueError(f"The demo history table already captures changes from {other[0]}")
        script = Path("order_history.sql").read_text(encoding="utf-8").strip()
        attach = "AFTER INSERT OR UPDATE OR DELETE ON public.orders\n"
        if not script.startswith("BEGIN;\n") or not script.endswith("\nCOMMIT;") or script.count(attach) != 1:
            raise ValueError("Use the matching order_history.sql from this Git folder.")
        body = script.removeprefix("BEGIN;\n").removesuffix("\nCOMMIT;")
        target = sql.SQL("AFTER INSERT OR UPDATE OR DELETE ON {}\n").format(source_table).as_string(db)
        db.execute(body.replace(attach, target))
        print(f"Installed the trigger on {source_name}.")

def read_rest(key, amount, kind):
    reply = http.get(api_url + "/public/orders_cdc", params={
        "select": "operation,old_amount_cents,new_amount_cents", "order_id": f"eq.{key}",
        "operation": "eq." + kind.upper(), "new_amount_cents": f"eq.{amount}",
        "order": "event_id.desc", "limit": 1,
    }, timeout=25)
    if reply.status_code in (401, 403):
        raise PermissionError(f"Lakebase Data API HTTP {reply.status_code}. Use an authenticated non-owner reader. "
                              "Grant the reader role to authenticator, USAGE on public, and SELECT on public.orders_cdc.")
    if reply.status_code == 404:
        raise LookupError("Expose public in the Data API and click Refresh schema cache after creating orders_cdc.")
    reply.raise_for_status()
    return reply.json()

def read_delta(key, amount, kind):
    event = "update_postimage" if kind == "update" else kind
    query = f"""SELECT _pg_change_type, amount_cents FROM {history}
                WHERE order_id=:key AND amount_cents=:amount AND _pg_change_type=:event
                ORDER BY _sort_by DESC LIMIT 1"""
    params = [dict(name="key", type="BIGINT", value=str(key)), dict(name="amount", type="BIGINT", value=str(amount)),
              dict(name="event", type="STRING", value=event)]
    route = "/api/2.0/sql/statements"
    response = w.api_client.do("POST", route, body={
        "warehouse_id": warehouse, "statement": query, "parameters": params,
        "wait_timeout": "10s", "on_wait_timeout": "CONTINUE",
    })
    deadline = time.monotonic() + 150
    while response["status"]["state"] in {"PENDING", "RUNNING"}:
        if time.monotonic() >= deadline:
            raise TimeoutError("SQL warehouse statement timed out")
        time.sleep(0.25)
        response = w.api_client.do("GET", route + "/" + response["statement_id"])
    if response["status"]["state"] != "SUCCEEDED":
        raise RuntimeError(response["status"].get("error", response["status"]))
    return response["result"].get("data_array", [])

def observe(read, key, amount, kind, expected, committed):
    deadline = committed + 150
    count = 0
    while time.perf_counter() < deadline:
        started = time.perf_counter()
        rows = read(key, amount, kind)
        finished = time.perf_counter()
        count += 1
        if rows:
            if rows != expected:
                raise ValueError(f"Unexpected {kind} change for {key}: {rows}")
            return {"commit_to_response_ms": (finished - committed) * 1000,
                    "successful_read_ms": (finished - started) * 1000, "requests": count}
        time.sleep(0.5)
    raise TimeoutError(f"{read.__name__} did not return the committed {kind} event")

read_rest(0, 0, "insert")
read_delta(0, 0, "insert")
print("Both readers can query their history table.")

In [ ]:
key = -(uuid4().int % (2**62) + 1)
previous = 714
measurements = []
with connect() as db, ThreadPoolExecutor(max_workers=2) as pool:
    db.execute(sql.SQL("INSERT INTO {} (order_id, tenant_id, amount_cents) VALUES (%s,10,%s)").format(source_table),
               (key, previous))
    db.commit()
    try:
        inserted = time.perf_counter()
        observe(read_rest, key, previous, "insert",
                [{"operation":"INSERT", "old_amount_cents":None, "new_amount_cents":previous}], inserted)
        observe(read_delta, key, previous, "insert", [["insert", str(previous)]], inserted)
        for amount in range(799, 804):
            read_rest(0, 0, "insert")
            read_delta(0, 0, "insert")
            update = sql.SQL("UPDATE {} SET amount_cents=%s WHERE order_id=%s AND amount_cents=%s").format(source_table)
            if db.execute(update, (amount, key, previous)).rowcount != 1:
                raise RuntimeError("The benchmark order changed unexpectedly")
            db.commit()
            committed = time.perf_counter()
            events = [
                ("Trigger + Data API", read_rest, [{"operation":"UPDATE", "old_amount_cents":previous, "new_amount_cents":amount}]),
                ("Lakebase CDF + SQL warehouse", read_delta, [["update_postimage", str(amount)]]),
            ]
            runs = [(label, pool.submit(observe, read, key, amount, "update", expected, committed))
                    for label, read, expected in events]
            for label, run in runs:
                measurements.append({"path": label, "old_amount": previous, "new_amount": amount, **run.result()})
            previous = amount
    finally:
        db.rollback()
        db.execute(sql.SQL("DELETE FROM {} WHERE order_id=%s").format(source_table), (key,))
        db.commit()

rows = pd.DataFrame(measurements)
display(rows)
display(rows.groupby("path")[["commit_to_response_ms","successful_read_ms","requests"]].median().round(2))